# Fine-tuning y Evaluación de BGE-M3 — Ampara AI

**Archivos necesarios** (cargar en `/content` antes de ejecutar):
- `training.json`
- `corpus.jsonl`
- `test_queries.jsonl`
- `test_qrels.jsonl`

**Flujo del notebook:**
1. Verificar GPU
2. Instalar dependencias
3. Configurar rutas e hiperparámetros
4. Explorar dataset
5. Corregir archivos de evaluación (IDs consistentes)
6. Fine-tuning con `torchrun` + módulo `encoder_only.m3`
7. Copiar tokenizer al checkpoint final
8. Definir funciones de evaluación
9. Evaluar modelo BASE
10. Evaluar modelo FINE-TUNED
11. Tabla comparativa de métricas
12. Guardar resultados

## 0. Verificar GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nCUDA disponible : {torch.cuda.is_available()}")
print(f"GPUs detectadas : {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU 0           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Mon Jun  8 12:10:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   56C    P0             30W /   72W |     262MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Instalación de dependencias

In [ ]:
!pip install -q -U \
    FlagEmbedding \
    faiss-cuda \
    pytrec_eval \
    datasets \
    transformers \
    accelerate \
    peft \
    deepspeed
print('Dependencias instaladas')

✓ Dependencias instaladas


In [ ]:
import json
import numpy as np
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

# Cargar todos los positivos del dataset
lengths = []
long_examples = []

with open("/content/training.json", "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        sample = json.loads(line)
        pos = sample["pos"]
        text = pos[0] if isinstance(pos, list) else pos
        tokens = tokenizer(text, truncation=False)["input_ids"]
        lengths.append(len(tokens))
        if len(tokens) > 512:
            long_examples.append((len(tokens), text[:150]))

lengths = np.array(lengths)

print(f"Total documentos analizados : {len(lengths)}")
print(f"Longitud media              : {lengths.mean():.0f} tokens")
print(f"Longitud mediana            : {np.median(lengths):.0f} tokens")
print(f"Percentil 75                : {np.percentile(lengths, 75):.0f} tokens")
print(f"Percentil 90                : {np.percentile(lengths, 90):.0f} tokens")
print(f"Percentil 95                : {np.percentile(lengths, 95):.0f} tokens")
print(f"Máximo                      : {lengths.max():.0f} tokens")
print(f"Documentos > 512 tokens     : {len(long_examples)} ({len(long_examples)/len(lengths)*100:.1f}%)")

if long_examples:
    print(f"\nEjemplos truncados (top 5 más largos):")
    for length, text in sorted(long_examples, reverse=True)[:5]:
        print(f"  [{length} tokens] {text}...")

Total documentos analizados : 810
Longitud media              : 485 tokens
Longitud mediana            : 314 tokens
Percentil 75                : 460 tokens
Percentil 90                : 1169 tokens
Percentil 95                : 1966 tokens
Máximo                      : 2251 tokens
Documentos > 512 tokens     : 201 (24.8%)

Ejemplos truncados (top 5 más largos):
  [2251 tokens] Artículo 10. Derecho a la asistencia y la protección integrales
Las entidades que conforman el Sistema Nacional para la Prevención, Sanción y Erradica...
  [2251 tokens] Artículo 10. Derecho a la asistencia y la protección integrales
Las entidades que conforman el Sistema Nacional para la Prevención, Sanción y Erradica...
  [2023 tokens] Artículo 16. Proceso
En el plazo máximo de setenta y dos horas, siguientes a la interposición de la denuncia, el juzgado de familia o su equivalente p...
  [2023 tokens] Artículo 16. Proceso
En el plazo máximo de setenta y dos horas, siguientes a la interposición de la denuncia,

## 2. Configuración de rutas e hiperparámetros

> Se ajusta `BATCH_SIZE` y `GRAD_ACCUM` para la GPU L4:
> - **L4  (~24 GB):** `BATCH_SIZE=1`, `GRAD_ACCUM=16`

In [ ]:
import os, json, torch, gc, shutil
from pathlib import Path
from collections import defaultdict

# ── Rutas ───────────────────────────────────────────────────────

DATA_DIR      = Path("/content")

TRAINING_FILE = DATA_DIR / "training.json"
CORPUS_FILE   = DATA_DIR / "corpus.jsonl"
QUERIES_FILE  = DATA_DIR / "test_queries.jsonl"
QRELS_FILE    = DATA_DIR / "test_qrels.jsonl"

# Archivos corregidos
CORPUS_FIXED  = DATA_DIR / "corpus_fixed.jsonl"
QRELS_FIXED   = DATA_DIR / "test_qrels_fixed.jsonl"

# ── Modelo ──────────────────────────────────────────────────────
BASE_MODEL    = "BAAI/bge-m3"
OUTPUT_DIR    = "/content/bge-m3-legal-peru"
CACHE_MODEL   = "/content/cache/model"
CACHE_DATA    = "/content/cache/data"

# ── Hiperparámetros ─────────────────────────────────────────────
NUM_EPOCHS    = 5
BATCH_SIZE    = 1
GRAD_ACCUM    = 16
LEARNING_RATE = 1e-5
MAX_LEN       = 2048
TEMPERATURE   = 0.02
TRAIN_GROUP_SIZE = 11
K_VALUES      = [10, 100]

N_GPUS = torch.cuda.device_count()
print(f"GPUs para entrenamiento: {N_GPUS}")
print(f"Batch efectivo         : {BATCH_SIZE * GRAD_ACCUM}")

print()
files_ok = True
for f in [TRAINING_FILE, CORPUS_FILE, QUERIES_FILE, QRELS_FILE]:
    exists = f.exists()
    print(f"  {'✓' if exists else '✗ NO ENCONTRADO'}  {f}")
    if not exists:
        files_ok = False

print()
print("Todos los archivos detectados.")

GPUs para entrenamiento: 1
Batch efectivo         : 16

  ✓  /content/training.json
  ✓  /content/corpus.jsonl
  ✓  /content/test_queries.jsonl
  ✓  /content/test_qrels.jsonl

✓ Todos los archivos detectados.


## 3. Exploración del dataset

In [ ]:
train_samples = []
with open(TRAINING_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            train_samples.append(json.loads(line))

sample = train_samples[0]
n_neg  = len(sample.get('neg', []))

print(f"Total ejemplos train  : {len(train_samples)}")
print(f"Negativos por query   : {n_neg}")
print(f"Campos disponibles    : {list(sample.keys())}")
print(f"\n── Ejemplo ──")
print(f"  query  : {sample['query'][:120]}")
print(f"  pos[0] : {str(sample['pos'])[:120]}")
print(f"  neg[0] : {sample['neg'][0][:120]}")

# Actualizar TRAIN_GROUP_SIZE según los negativos reales
TRAIN_GROUP_SIZE = n_neg + 1
print(f"\n→ TRAIN_GROUP_SIZE = {TRAIN_GROUP_SIZE} ({n_neg} negativos + 1 positivo)")

Total ejemplos train  : 810
Negativos por query   : 10
Campos disponibles    : ['id', 'query', 'pos', 'neg', 'prompt', 'dataset_id', 'case_id', 'topic', 'barrier']

── Ejemplo ──
  query  : ¿Lo que me pasó con el vecino cuenta como abuso sexual aunque solo me tocó?
  pos[0] : ['Artículo 1. Objeto de la Ley\nLa presente Ley tiene por objeto prevenir, erradicar y sancionar toda forma de violencia
  neg[0] : Título: Que la declaración de la víctima en cámara Gesell no se haya recabado como prueba anticipada no implica que dich

→ TRAIN_GROUP_SIZE = 11 (10 negativos + 1 positivo)


## 4. Corrección de archivos de evaluación

El script de preparación genera IDs con sufijo (`247_0`, `247_1`) cuando un positivo
tiene múltiples textos. Esto rompe el mapeo `qid → docid` en la evaluación porque
las queries tienen IDs simples (`247`) pero el corpus tiene IDs con sufijo.

Esta sección reconstruye `corpus_fixed.jsonl` y `test_qrels_fixed.jsonl` con IDs consistentes.

In [ ]:
from datasets import load_dataset

# ── 1. Corpus nuevo: un documento por ID, sin sufijos ────────────
corpus_new = {}
for s in train_samples:
    sid  = str(s["id"])
    pos  = s["pos"]
    text = pos[0] if isinstance(pos, list) else pos
    if sid not in corpus_new:
        corpus_new[sid] = text

# ── 2. Agregar positivos de las queries de test ──────────────────
# (pueden no estar en training.json si el split fue aleatorio)
test_queries_ds = load_dataset("json", data_files=str(QUERIES_FILE))["train"]
qrels_ds_orig   = load_dataset("json", data_files=str(QRELS_FILE))["train"]
corpus_ds_orig  = load_dataset("json", data_files=str(CORPUS_FILE))["train"]

id_to_text_orig = {str(r["id"]): r["text"] for r in corpus_ds_orig}

qid_to_docids = defaultdict(list)
for row in qrels_ds_orig:
    qid_to_docids[str(row["qid"])].append(str(row["docid"]))

for row in test_queries_ds:
    qid = str(row["id"])
    if qid not in corpus_new:
        for docid in qid_to_docids.get(qid, []):
            if docid in id_to_text_orig:
                corpus_new[qid] = id_to_text_orig[docid]
                break

print(f"Corpus corregido: {len(corpus_new)} documentos")

# ── 3. Guardar corpus corregido ──────────────────────────────────
with open(CORPUS_FIXED, "w", encoding="utf-8") as f:
    for sid, text in corpus_new.items():
        f.write(json.dumps({"id": sid, "text": text}, ensure_ascii=False) + "\n")

# ── 4. Qrels corregidos: qid → qid (ID simple a ID simple) ──────
qrels_new = []
missing   = []
for row in test_queries_ds:
    qid = str(row["id"])
    if qid in corpus_new:
        qrels_new.append({"qid": qid, "docid": qid, "relevance": 1})
    else:
        missing.append(qid)

if missing:
    print(f"⚠️  {len(missing)} queries sin positivo en corpus: {missing[:5]}")

with open(QRELS_FIXED, "w", encoding="utf-8") as f:
    for item in qrels_new:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

# ── 5. Verificación ──────────────────────────────────────────────
corpus_ids_new  = set(corpus_new.keys())
qrel_docids_new = {r["docid"] for r in qrels_new}
overlap         = corpus_ids_new & qrel_docids_new

print(f"Queries         : {len(test_queries_ds)}")
print(f"Docs en corpus  : {len(corpus_ids_new)}")
print(f"Qrels           : {len(qrels_new)}")
print(f"Overlap docids  : {len(overlap)} de {len(qrel_docids_new)}")
status = '✓ IDs consistentes — listo para evaluar' if len(overlap) == len(qrel_docids_new) else '✗ Aún hay IDs sin match'
print(f"\n{status}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Corpus corregido: 900 documentos
Queries         : 90
Docs en corpus  : 900
Qrels           : 90
Overlap docids  : 90 de 90

✓ IDs consistentes — listo para evaluar


## 5. Fine-tuning con `torchrun`

Módulo: `FlagEmbedding.finetune.embedder.encoder_only.m3`

- **InfoNCE loss** con negativos in-batch + negativos explícitos del dataset
- `unified_finetuning=False` optimiza dense + sparse + colbert pero requiere mas de 24GB VRAM


### DeepSpeed Stage 0

In [ ]:
ds_config = {
    "train_micro_batch_size_per_gpu": "auto",
    "gradient_accumulation_steps": "auto",
    "gradient_clipping": "auto",
    "zero_optimization": {"stage": 0},
    "fp16": {
        "enabled": "auto",
        "loss_scale": 0,
        "loss_scale_window": 1000,
        "initial_scale_power": 16,
        "hysteresis": 2,
        "min_loss_scale": 1
    },
    "steps_per_print": 10,
    "wall_clock_breakdown": False
}

ds_config_path = "/content/ds_stage0.json"
with open(ds_config_path, "w") as f:
    json.dump(ds_config, f, indent=2)

print(f"DeepSpeed config guardado en: {ds_config_path}")

✓ DeepSpeed config guardado en: /content/ds_stage0.json


In [ ]:
# Calcular warmup_steps (10% de los pasos totales)
steps_per_epoch = max(1, len(train_samples) // (BATCH_SIZE * GRAD_ACCUM))
total_steps     = steps_per_epoch * NUM_EPOCHS
warmup_steps    = max(1, int(total_steps * 0.1))
# Checkpoint cada epoch completo
save_steps      = steps_per_epoch

print(f"Pasos por epoch : {steps_per_epoch}")
print(f"Pasos totales   : {total_steps}")
print(f"Warmup steps    : {warmup_steps}")
print(f"Save cada       : {save_steps} pasos (1 epoch)")

# Limpiar OUTPUT_DIR si ya existe
if Path(OUTPUT_DIR).exists():
    shutil.rmtree(OUTPUT_DIR)
    print(f"\n✓ Directorio {OUTPUT_DIR} limpiado.")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"\nVRAM detectada  : {vram_gb:.1f} GB")

ft_cmd = f"""\
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \\
torchrun \\
    --nproc_per_node {N_GPUS} \\
    -m FlagEmbedding.finetune.embedder.encoder_only.m3 \\
    --model_name_or_path {BASE_MODEL} \\
    --cache_dir {CACHE_MODEL} \\
    --train_data {TRAINING_FILE} \\
    --cache_path {CACHE_DATA} \\
    --train_group_size {TRAIN_GROUP_SIZE} \\
    --query_max_len {MAX_LEN} \\
    --passage_max_len {MAX_LEN} \\
    --pad_to_multiple_of 8 \\
    --same_dataset_within_batch True \\
    --small_threshold 0 \\
    --drop_threshold 0 \\
    --output_dir {OUTPUT_DIR} \\
    --learning_rate {LEARNING_RATE} \\
    --fp16 \\
    --num_train_epochs {NUM_EPOCHS} \\
    --per_device_train_batch_size {BATCH_SIZE} \\
    --gradient_accumulation_steps {GRAD_ACCUM} \\
    --dataloader_drop_last True \\
    --warmup_steps {warmup_steps} \\
    --gradient_checkpointing \\
    --deepspeed {ds_config_path} \\
    --logging_steps 10 \\
    --save_steps {save_steps} \\
    --negatives_cross_device \\
    --temperature {TEMPERATURE} \\
    --sentence_pooling_method cls \\
    --normalize_embeddings True \\
    --unified_finetuning False \\
    --use_self_distill False \\
    --fix_encoder False \\
    --self_distill_start_step {warmup_steps}\
"""

print("\nComando de fine-tuning:")
print("─" * 60)
print(ft_cmd)

Pasos por epoch : 50
Pasos totales   : 250
Warmup steps    : 25
Save cada       : 50 pasos (1 epoch)

✓ Directorio /content/bge-m3-legal-peru limpiado.

VRAM detectada  : 23.7 GB
unified_finetuning: True

Comando de fine-tuning:
────────────────────────────────────────────────────────────
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
torchrun \
    --nproc_per_node 1 \
    -m FlagEmbedding.finetune.embedder.encoder_only.m3 \
    --model_name_or_path BAAI/bge-m3 \
    --cache_dir /content/cache/model \
    --train_data /content/training.json \
    --cache_path /content/cache/data \
    --train_group_size 11 \
    --query_max_len 2048 \
    --passage_max_len 2048 \
    --pad_to_multiple_of 8 \
    --same_dataset_within_batch True \
    --small_threshold 0 \
    --drop_threshold 0 \
    --output_dir /content/bge-m3-legal-peru \
    --learning_rate 1e-05 \
    --fp16 \
    --num_train_epochs 5 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 16 \
    --dataloa

### Correccion de modulo FlagEmbedding para solucionar el error de mapeo de tokenizador incorrecto al guardar cada checkpoint

In [ ]:
import FlagEmbedding.finetune.embedder.encoder_only.m3.trainer as m3_trainer
import inspect, textwrap
trainer_path = inspect.getfile(m3_trainer)
print(f"Archivo a parchear: {trainer_path}")

with open(trainer_path, "r") as f:
    content = f.read()
if "self.tokenizer = self.model.tokenizer" in content:
    print("✓ El fix ya está aplicado.")
else:
    # El fix: en _save, antes de usar self.tokenizer,
    # obtenerlo del modelo si no existe
    old = "    def _save(self, output_dir"
    new = """    def _save(self, output_dir"""

    fix = '''    def _save(self, output_dir=None, state_dict=None):
        # Fix: FlagEmbedding bug — tokenizer not set on trainer
        if not hasattr(self, 'tokenizer') or self.tokenizer is None:
            try:
                from transformers import AutoTokenizer
                model_name = self.args.model_name_or_path if hasattr(self.args, 'model_name_or_path') else "BAAI/bge-m3"
                self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            except Exception:
                self.tokenizer = None
        return super()._save(output_dir=output_dir, state_dict=state_dict)

'''
    if "def _save(" in content:
        import re
        content_new = re.sub(
            r'    def _save\(self[^)]*\):',
            '    def _save(self, output_dir=None, state_dict=None):\n'
            '        if not hasattr(self, \'tokenizer\') or self.tokenizer is None:\n'
            '            try:\n'
            '                from transformers import AutoTokenizer\n'
            '                self.tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")\n'
            '            except Exception:\n'
            '                self.tokenizer = None',
            content,
            count=1
        )
        with open(trainer_path, "w") as f:
            f.write(content_new)
        print("✓ Fix aplicado al trainer de FlagEmbedding.")
    else:
        print("No se encontró _save en el trainer. Aplicando fix alternativo...")
        content_new = content + "\n" + fix
        with open(trainer_path, "w") as f:
            f.write(content_new)
        print("✓ Fix aplicado (método agregado).")

print("\nContenido relevante del trainer tras el fix:")
with open(trainer_path, "r") as f:
    for i, line in enumerate(f):
        if "_save" in line or "tokenizer" in line:
            print(f"  línea {i+1}: {line.rstrip()}")

Archivo a parchear: /usr/local/lib/python3.12/dist-packages/FlagEmbedding/finetune/embedder/encoder_only/m3/trainer.py
✓ Fix aplicado al trainer de FlagEmbedding.

Contenido relevante del trainer tras el fix:
  línea 15:     def _save(self, output_dir=None, state_dict=None):
  línea 16:         if not hasattr(self, 'tokenizer') or self.tokenizer is None:
  línea 19:                 self.tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
  línea 21:                 self.tokenizer = None
  línea 41:         if self.tokenizer is not None and self.is_world_process_zero():
  línea 42:             self.tokenizer.save_pretrained(output_dir)


In [ ]:
import subprocess

print("Iniciando fine-tuning...")
print(f"   Epochs          : {NUM_EPOCHS}")
print(f"   Batch/GPU       : {BATCH_SIZE}  (efectivo: {BATCH_SIZE * GRAD_ACCUM})")
print(f"   Group size      : {TRAIN_GROUP_SIZE}")
print(f"   GPUs            : {N_GPUS}")
print(f"   unified_ft      : {use_unified}")
print("=" * 60)

process = subprocess.Popen(
    ft_cmd,
    shell=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()

if process.returncode == 0:
    print("\n✓ Fine-tuning completado exitosamente.")
else:
    print(f"\n✗ Error en fine-tuning (código {process.returncode}).")

🚀 Iniciando fine-tuning...
   Epochs          : 5
   Batch/GPU       : 1  (efectivo: 16)
   Group size      : 11
   GPUs            : 1
   unified_ft      : True
06/08/2026 12:11:21 - WARNING - FlagEmbedding.abc.finetune.embedder.AbsRunner -   Process rank: -1, device: cuda:0, n_gpu: 1, distributed training: False, 16-bits training: True
06/08/2026 12:11:21 - INFO - FlagEmbedding.abc.finetune.embedder.AbsRunner -   Training/evaluation parameters EncoderOnlyEmbedderM3TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=True,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_

## Guardar el ultimo checkpoint en google drive

In [ ]:
from pathlib import Path
import shutil
from google.colab import drive
drive.mount('/content/drive')

# Ver qué hay en FT_MODEL_PATH
print(f"Archivos en {FT_MODEL_PATH}:")
for f in sorted(Path(FT_MODEL_PATH).iterdir()):
    if f.is_file():
        print(f"  {f.name:<45} {f.stat().st_size/1e6:.1f} MB")

# Copiar solo los 6 archivos necesarios a Drive
DEST = Path("/content/drive/MyDrive/bge-m3-legal-peru-2048/modelo_final")
DEST.mkdir(parents=True, exist_ok=True)

archivos_necesarios = [
    "model.safetensors",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "sentencepiece.bpe.model",
    "special_tokens_map.json",
]

for nombre in archivos_necesarios:
    src = Path(FT_MODEL_PATH) / nombre
    if src.exists():
        shutil.copy2(src, DEST / nombre)
        print(f"  ✓ {nombre}  ({src.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"  ✗ {nombre}  — no encontrado, buscando alternativa...")
        # Buscar en global_step255 como fallback
        alt = Path(FT_MODEL_PATH) / "global_step255" / nombre
        if alt.exists():
            shutil.copy2(alt, DEST / nombre)
            print(f"    → encontrado en global_step255")

print(f"\n✓ Modelo guardado en Drive: {DEST}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archivos en /content/bge-m3-legal-peru/checkpoint-255:
  config.json                                   0.0 MB
  latest                                        0.0 MB
  model.safetensors                             1135.6 MB
  rng_state.pth                                 0.0 MB
  scheduler.pt                                  0.0 MB
  tokenizer.json                                17.1 MB
  tokenizer_config.json                         0.0 MB
  trainer_state.json                            0.0 MB
  training_args.bin                             0.0 MB
  ✓ model.safetensors  (1135.6 MB)
  ✓ config.json  (0.0 MB)
  ✓ tokenizer.json  (17.1 MB)
  ✓ tokenizer_config.json  (0.0 MB)
  ✗ sentencepiece.bpe.model  — no encontrado, buscando alternativa...
  ✗ special_tokens_map.json  — no encontrado, buscando alternativa...

✓ Modelo guardado en Drive: /content/drive/MyDriv

## 6. Preparar el modelo fine-tuned para evaluación

El trainer guarda el modelo como `checkpoint-N`
Detectamos el checkpoint final y copiamos el tokenizer desde el modelo base
por el bug del trainer que no guardar el tokenizador correctamente


In [ ]:
from transformers import AutoTokenizer, AutoConfig

# Detectar el checkpoint con el número más alto
output_path    = Path(OUTPUT_DIR)
checkpoints    = sorted(
    [d for d in output_path.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")],
    key=lambda d: int(d.name.split("-")[1])
)

if not checkpoints:
    raise RuntimeError(f"No se encontraron checkpoints en {OUTPUT_DIR}")

FT_MODEL_PATH = str(checkpoints[-1])
print(f"Checkpoint final detectado: {FT_MODEL_PATH}")
print(f"Todos los checkpoints: {[c.name for c in checkpoints]}")

# Copiar tokenizer y config del modelo base al checkpoint final
print("\nCopiando tokenizer desde el modelo base...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.save_pretrained(FT_MODEL_PATH)

config = AutoConfig.from_pretrained(BASE_MODEL)
config.save_pretrained(FT_MODEL_PATH)

# Verificar archivos
print(f"\nArchivos en {Path(FT_MODEL_PATH).name}:")
for f in sorted(Path(FT_MODEL_PATH).iterdir()):
    if f.is_file():
        print(f"  {f.name:<45} {f.stat().st_size/1e6:.2f} MB")

critical = any(f.name in ["model.safetensors", "pytorch_model.bin"]
               for f in Path(FT_MODEL_PATH).iterdir())
print(f"\n{'✓ Modelo listo para evaluación' if critical else '✗ Faltan pesos del modelo'}")

Checkpoint final detectado: /content/bge-m3-legal-peru/checkpoint-255
Todos los checkpoints: ['checkpoint-50', 'checkpoint-100', 'checkpoint-150', 'checkpoint-200', 'checkpoint-250', 'checkpoint-255']

Copiando tokenizer desde el modelo base...

Archivos en checkpoint-255:
  config.json                                   0.00 MB
  latest                                        0.00 MB
  model.safetensors                             1135.55 MB
  rng_state.pth                                 0.01 MB
  scheduler.pt                                  0.00 MB
  tokenizer.json                                17.10 MB
  tokenizer_config.json                         0.00 MB
  trainer_state.json                            0.01 MB
  training_args.bin                             0.01 MB

✓ Modelo listo para evaluación


## 7. Funciones de evaluación

In [ ]:
import numpy as np
import faiss
from tqdm.notebook import tqdm
from datasets import load_dataset
from FlagEmbedding import BGEM3FlagModel
from FlagEmbedding.abc.evaluation.utils import evaluate_metrics, evaluate_mrr


def load_eval_data(corpus_file, queries_file, qrels_file):
    """Carga los tres archivos de evaluación con IDs corregidos."""
    queries_ds = load_dataset("json", data_files=str(queries_file))["train"]
    corpus_ds  = load_dataset("json", data_files=str(corpus_file))["train"]
    qrels_ds   = load_dataset("json", data_files=str(qrels_file))["train"]

    queries_text = queries_ds["text"]

    corpus_text = []
    for item in corpus_ds["text"]:
        if isinstance(item, list):
            corpus_text.extend(item)
        else:
            corpus_text.append(item)

    qrels_dict = defaultdict(dict)
    for row in qrels_ds:
        qrels_dict[str(row["qid"])][str(row["docid"])] = int(row["relevance"])

    print(f"  Queries : {len(queries_text)}")
    print(f"  Corpus  : {len(corpus_text)} documentos")
    print(f"  Qrels   : {len(qrels_dict)} queries anotadas")
    return queries_ds, corpus_ds, queries_text, corpus_text, qrels_dict


def encode_and_search(model, queries_ds, corpus_ds,
                      queries_text, corpus_text, k=100, batch_size=32):
    """Codifica con BGE-M3 y busca top-k con FAISS."""
    print(f"  Codificando {len(queries_text)} queries...")
    q_out = model.encode(
        queries_text, batch_size=batch_size, max_length=512, return_dense=True
    )
    queries_emb = np.array(q_out["dense_vecs"], dtype=np.float32)

    print(f"  Codificando {len(corpus_text)} documentos...")
    c_out = model.encode(
        corpus_text, batch_size=batch_size, max_length=512, return_dense=True
    )
    corpus_emb = np.array(c_out["dense_vecs"], dtype=np.float32)

    dim   = corpus_emb.shape[-1]
    index = faiss.index_factory(dim, "Flat", faiss.METRIC_INNER_PRODUCT)
    index.train(corpus_emb)
    index.add(corpus_emb)

    print(f"  Buscando top-{k}...")
    all_scores, all_indices = [], []
    for i in tqdm(range(0, len(queries_emb), batch_size), desc="  Searching"):
        j = min(i + batch_size, len(queries_emb))
        scores, indices = index.search(queries_emb[i:j], k=k)
        all_scores.append(scores)
        all_indices.append(indices)

    all_scores  = np.concatenate(all_scores,  axis=0)
    all_indices = np.concatenate(all_indices, axis=0)

    results = {}
    for idx, (scores, indices) in enumerate(zip(all_scores, all_indices)):
        qid = str(queries_ds["id"][idx])
        results[qid] = {}
        for score, doc_idx in zip(scores, indices):
            if doc_idx != -1:
                docid = str(corpus_ds["id"][doc_idx])
                results[qid][docid] = float(score)
    return results


def run_eval(model, label, queries_ds, corpus_ds,
             queries_text, corpus_text, qrels_dict, k_values, batch_size=32):
    """Búsqueda + métricas. Devuelve dict con todos los valores."""
    print(f"\n{'─'*55}")
    print(f"  Evaluando: {label}")
    print(f"{'─'*55}")

    results  = encode_and_search(
        model, queries_ds, corpus_ds,
        queries_text, corpus_text,
        k=max(k_values), batch_size=batch_size
    )
    eval_res = evaluate_metrics(qrels_dict, results, k_values)
    mrr_res  = evaluate_mrr(qrels_dict, results, k_values)

    all_metrics = {}
    print(f"\n  Métricas:")
    for metric_dict in eval_res:
        for key, val in metric_dict.items():
            print(f"    {key:<20} {val:.5f}")
            all_metrics[key] = val
    for key, val in mrr_res.items():
        print(f"    {key:<20} {val:.5f}")
        all_metrics[key] = val
    return all_metrics


def print_comparison(base_metrics, ft_metrics):
    print(f"\n{'='*65}")
    print(f"  COMPARACIÓN — Base vs Fine-tuned (bge-m3 legal perú)")
    print(f"{'='*65}")
    print(f"  {'Métrica':<22} {'Base':>10} {'Fine-tuned':>12} {'Delta':>8} {'Mejora':>8}")
    print(f"  {'─'*22} {'─'*10} {'─'*12} {'─'*8} {'─'*8}")
    for key in sorted(set(base_metrics) | set(ft_metrics)):
        b   = base_metrics.get(key, 0.0)
        ft  = ft_metrics.get(key, 0.0)
        d   = ft - b
        pct = (d / b * 100) if b > 0 else 0.0
        arrow = "↑" if d > 0 else ("↓" if d < 0 else "=")
        print(f"  {key:<22} {b:>10.5f} {ft:>12.5f} {arrow}{abs(d):>6.5f} {pct:>+7.1f}%")
    print(f"{'='*65}")


print("Funciones de evaluación listas.")

✓ Funciones de evaluación listas.


## 8. Cargar datos de evaluación (con IDs corregidos)

In [ ]:
print("Cargando datos de evaluación (archivos corregidos)...")
queries_ds, corpus_ds, queries_text, corpus_text, qrels_dict = load_eval_data(
    CORPUS_FIXED, QUERIES_FILE, QRELS_FIXED
)

Cargando datos de evaluación (archivos corregidos)...


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

  Queries : 90
  Corpus  : 900 documentos
  Qrels   : 90 queries anotadas


## 9. Evaluar modelo BASE (antes del fine-tuning)

In [ ]:
print(f"Cargando {BASE_MODEL}...")
base_model = BGEM3FlagModel(BASE_MODEL, use_fp16=True)

base_metrics = run_eval(
    base_model,
    label=f"BASE — {BASE_MODEL}",
    queries_ds=queries_ds,
    corpus_ds=corpus_ds,
    queries_text=queries_text,
    corpus_text=corpus_text,
    qrels_dict=qrels_dict,
    k_values=K_VALUES,
    batch_size=32,
)

del base_model
gc.collect()
torch.cuda.empty_cache()
print("\n✓ Memoria GPU liberada.")

Cargando BAAI/bge-m3...


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


───────────────────────────────────────────────────────
  Evaluando: BASE — BAAI/bge-m3
───────────────────────────────────────────────────────
  Codificando 90 queries...


Inference Embeddings: 100%|██████████| 3/3 [00:00<00:00, 49.06it/s]


  Codificando 900 documentos...


Inference Embeddings: 100%|██████████| 29/29 [00:05<00:00,  5.04it/s]

  Buscando top-100...


  Searching:   0%|          | 0/3 [00:00<?, ?it/s]


  Métricas:
    NDCG@10              0.08510
    NDCG@100             0.13103
    MAP@10               0.07001
    MAP@100              0.07688
    Recall@10            0.13333
    Recall@100           0.37778
    P@10                 0.01333
    P@100                0.00378
    MRR@10               0.08373
    MRR@100              0.09124

✓ Memoria GPU liberada.


## 10. Evaluar modelo FINE-TUNED (después del entrenamiento)

In [ ]:
print(f"Cargando modelo fine-tuned desde: {FT_MODEL_PATH}")
ft_model = BGEM3FlagModel(FT_MODEL_PATH, use_fp16=True)

ft_metrics = run_eval(
    ft_model,
    label=f"FINE-TUNED — {Path(FT_MODEL_PATH).name}",
    queries_ds=queries_ds,
    corpus_ds=corpus_ds,
    queries_text=queries_text,
    corpus_text=corpus_text,
    qrels_dict=qrels_dict,
    k_values=K_VALUES,
    batch_size=32,
)

del ft_model
gc.collect()
torch.cuda.empty_cache()
print("\n✓ Memoria GPU liberada.")

Cargando modelo fine-tuned desde: /content/bge-m3-legal-peru/checkpoint-255


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


───────────────────────────────────────────────────────
  Evaluando: FINE-TUNED — checkpoint-255
───────────────────────────────────────────────────────
  Codificando 90 queries...


Inference Embeddings: 100%|██████████| 3/3 [00:00<00:00, 45.70it/s]


  Codificando 900 documentos...


Inference Embeddings: 100%|██████████| 29/29 [00:05<00:00,  5.07it/s]

  Buscando top-100...


  Searching:   0%|          | 0/3 [00:00<?, ?it/s]


  Métricas:
    NDCG@10              0.11184
    NDCG@100             0.18522
    MAP@10               0.08475
    MAP@100              0.09636
    Recall@10            0.20000
    Recall@100           0.57778
    P@10                 0.02000
    P@100                0.00578
    MRR@10               0.13887
    MRR@100              0.15250

✓ Memoria GPU liberada.


## 11. Tabla comparativa final

In [ ]:
print_comparison(base_metrics, ft_metrics)


  COMPARACIÓN — Base vs Fine-tuned (bge-m3 legal perú)
  Métrica                      Base   Fine-tuned    Delta   Mejora
  ────────────────────── ────────── ──────────── ──────── ────────
  MAP@10                    0.07001      0.08475 ↑0.01474   +21.1%
  MAP@100                   0.07688      0.09636 ↑0.01948   +25.3%
  MRR@10                    0.08373      0.13887 ↑0.05514   +65.9%
  MRR@100                   0.09124      0.15250 ↑0.06126   +67.1%
  NDCG@10                   0.08510      0.11184 ↑0.02674   +31.4%
  NDCG@100                  0.13103      0.18522 ↑0.05419   +41.4%
  P@10                      0.01333      0.02000 ↑0.00667   +50.0%
  P@100                     0.00378      0.00578 ↑0.00200   +52.9%
  Recall@10                 0.13333      0.20000 ↑0.06667   +50.0%
  Recall@100                0.37778      0.57778 ↑0.20000   +52.9%


## 12. Guardar resultados y modelo

In [ ]:
eval_results = {
    "base_model":      base_metrics,
    "finetuned_model": ft_metrics,
    "delta":           {k: ft_metrics.get(k, 0) - base_metrics.get(k, 0)
                        for k in base_metrics},
    "config": {
        "base_model":       BASE_MODEL,
        "ft_checkpoint":    FT_MODEL_PATH,
        "num_epochs":       NUM_EPOCHS,
        "batch_size":       BATCH_SIZE,
        "grad_accum":       GRAD_ACCUM,
        "effective_batch":  BATCH_SIZE * GRAD_ACCUM,
        "learning_rate":    LEARNING_RATE,
        "temperature":      TEMPERATURE,
        "train_group_size": TRAIN_GROUP_SIZE,
        "unified_ft":       use_unified,
        "train_samples":    len(train_samples),
    }
}

results_path = "/content/eval_results.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)

print(f"✓ Métricas guardadas en: {results_path}")

✓ Métricas guardadas en: /content/eval_results.json
